In [124]:
import time
import requests
import networkx as nx
import seaborn as sns
from pyvis.network import Network
from matplotlib.colors import to_hex

sns.color_palette('colorblind')

[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [23]:
def get_details_given_title(title):
    title_match_url = "https://api.semanticscholar.org/graph/v1/paper/search/match"
    params = {"query": title, "fields": "paperId,title,year,authors,venue,publicationTypes,citationCount,referenceCount,influentialCitationCount"}
    response = requests.get(title_match_url, params=params)
    return response.json()

In [165]:
def get_references_of_paper(paper_id):
    base_url = "https://api.semanticscholar.org/graph/v1/paper/"
    response = requests.get(f'{base_url}{paper_id}/references')

    refernces = {}
    if 'data' in response.json():
        if response.json()['data']:
            for x in response.json()['data']:
                refernces[x['citedPaper']['paperId']] = x['citedPaper']['title']

    return refernces
        

In [166]:
def get_citations_by_paper(paper_id):
    base_url = "https://api.semanticscholar.org/graph/v1/paper/"
    response = requests.get(f'{base_url}{paper_id}/citations')

    citations = {}
    if 'data' in response.json():
        if response.json()['data']:
            for x in response.json()['data']:
                citations[x['citingPaper']['paperId']] = x['citingPaper']['title']

    return citations

In [160]:
collection_title = 'lgbtqia_data_donations'
collection_path = '/Users/brahmaninutakki/saarland/myconnectedpapeers/collections'

with open(f'{collection_path}/{collection_title}.txt', 'r') as f:
    paper_titles = f.readlines()

paper_titles

['Social Media Use and Health and Well-being of Lesbian, Gay, Bisexual, Transgender, and Queer Youth: Systematic Review\n',
 'Social media use and experiences among transgender and gender diverse adolescents\n',
 'Mapping gender transition sentiment patterns via social media data: toward decreasing transgender mental health disparities']

In [167]:
citations = {}
references = {}
papers = {}
for x in paper_titles:
    x = x.strip()
    papers[x] = get_details_given_title(x)
    paper_id = papers[x]['data'][0]['paperId']
    time.sleep(2)
    citations[x] = get_citations_by_paper(paper_id)
    time.sleep(2)
    references[x] = get_references_of_paper(paper_id)
    time.sleep(2)

In [168]:
papers, citations, references

({'Social Media Use and Health and Well-being of Lesbian, Gay, Bisexual, Transgender, and Queer Youth: Systematic Review': {'data': [{'paperId': 'd65c32cd8f300a8e93706ce283521842c2d67e04',
     'title': 'Social Media Use and Health and Well-being of Lesbian, Gay, Bisexual, Transgender, and Queer Youth: Systematic Review',
     'venue': 'Journal of Medical Internet Research',
     'year': 2022,
     'referenceCount': 59,
     'citationCount': 112,
     'influentialCitationCount': 3,
     'publicationTypes': ['Review', 'JournalArticle'],
     'authors': [{'authorId': '2147296614', 'name': 'M. Berger'},
      {'authorId': '32215607', 'name': 'Melody Taba'},
      {'authorId': '5793582', 'name': 'J. Marino'},
      {'authorId': '2067658590', 'name': 'Megan S. C. Lim'},
      {'authorId': '120935406', 'name': 'S. Skinner'}],
     'matchScore': 299.57812}]},
  'Social media use and experiences among transgender and gender diverse adolescents': {'data': [{'paperId': '71e0f8311bc1972183ff27970

In [172]:
net = Network(height="100vh", width="100%", directed=True, bgcolor="#ffffff", font_color="#222", cdn_resources='in_line')
net.force_atlas_2based()
node_size = 30
edge_weight = 10
sns_colors = [to_hex(c) for c in sns.color_palette("colorblind", n_colors=10)]

for paper in paper_titles:

    paper = paper.strip()
    net.add_node(
        paper, 
        label=paper, 
        title=paper, 
        shape="dot", 
        size=node_size + 10, 
        color={"background": sns_colors[3], "border": sns_colors[3]}
    )

    for id_, title_ in citations[paper].items():
        title_ = title_.strip()
        if title_ not in net.get_nodes():
            net.add_node(
                title_, 
                label=' ',
                title=title_, 
                shape="dot", 
                size=node_size, 
                color={"background": sns_colors[2], "border": sns_colors[2]}
            )
        net.add_edge(title_, paper, color="#2ca02c", weight=edge_weight)

    for id_, title_ in references[paper].items():
        title_ = title_.strip()
        if title_ not in net.get_nodes():
            net.add_node(
                title_, 
                label=' ', 
                title=title_, 
                shape="dot", 
                size=node_size, 
                color={"background": sns_colors[5], "border": sns_colors[5]}
            )
        net.add_edge(paper, title_, color="#2ca02c", weight=edge_weight)

# connected_nodes = {}
# for edge in net.get_edges():
#     node = edge['from']
#     to_node = edge['to']
#     if node not in connected_nodes:
#         connected_nodes[node] = 0
#     connected_nodes[node] += 1
    
#     node = edge['to']
#     if node not in connected_nodes:
#         connected_nodes[node] = 0
#     connected_nodes[node] += 1
    

# net.nodes = [n for n in net.nodes if connected_nodes.get(n['id'], 0) > 1]

remaining_ids = {n['id'] for n in net.nodes}
degree = {nid: 0 for nid in remaining_ids}



for e in net.get_edges():
    u, v = e['from'], e['to']
    if u in remaining_ids and v in remaining_ids:
        degree[u] += 1
        degree[v] += 1

min_visual_size = 20 
max_visual_size = 60 

deg_vals = list(degree.values()) or [1]
dmin, dmax = min(deg_vals), max(deg_vals)

def scale_size(d):
    if dmax == dmin:
        return (min_visual_size + max_visual_size) / 2
    return min_visual_size + (max_visual_size - min_visual_size) * (d - dmin) / (dmax - dmin)

net.nodes = [n for n in net.nodes if degree.get(n['id'], 0) > 1]

for n in net.nodes:
    n['size'] = scale_size(degree.get(n['id'], 0))

net.write_html('output.html')

In [171]:
sorted_degrees = sorted(degree.items(), key=lambda x: x[1], reverse=True)
sorted_degrees

[('Social Media Use and Health and Well-being of Lesbian, Gay, Bisexual, Transgender, and Queer Youth: Systematic Review',
  159),
 ('Mapping gender transition sentiment patterns via social media data: toward decreasing transgender mental health disparities',
  102),
 ('Social media use and experiences among transgender and gender diverse adolescents',
  21),
 ('Medical Information Provided by Transgender and Gender-Diverse Content Creators on YouTube: Descriptive Content Analysis',
  2),
 ('Social Media Creations of Community and Gender Minority Stress in Transgender and Gender-Diverse Adults',
  2),
 ('How social is social media for transgender and gender-diverse youth? Association of online social experiences with internalizing mental health problems',
  2),
 ('Trans-Jugendliche: aktuelle Entwicklungen in der Forschung', 2),
 ('Queer and trans influential peers: Negotiating platforms for community care',
  2),
 ('Social Media and Social Connection in Sexual and Gender Minority Young